# core

> Fill in a module description here

In [2]:
# | default_exp api

In [3]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [4]:
# | export
from typing import Any, List, TypedDict, Tuple, Sequence
from uuid import UUID
from product_horse.filesystems import (
    AbstractFileSystem,
    FilePathType,
)
from product_horse.db import (
    AbstractDatabase,
    UnvalidatedUser,
    UnvalidatedFileMetadata,
    User,
    FileMetadata,
    UnvalidatedTranscription,
    Transcription,
    Schema,
    UnvalidatedSchema,
    UnvalidatedUtterance,
    Utterance,
)
from product_horse.audio_video import AudioEditor, AssemblyAiTranscript
from product_horse.extraction import AIModelClient, Questions, Answers
from product_horse.search import SearchEngine


In [5]:
# | export


class FileDict(TypedDict):
    content: bytes
    name: str


def create_user_and_add_files(
    user_data: dict[str, Any],
    files: List[FileDict],
    db: AbstractDatabase,
    fs: AbstractFileSystem,
    authorized: bool = True,  # Is the user authorized to save these files?
) -> Tuple[User, List[FileMetadata]]:
    """
    Create a new user and add files for the user.

    Args:
        user_data (dict[str, Any]): User data dictionary.
        files (List[FileDict]): List of file dictionaries containing content and name.
        db (AbstractDatabase): Database instance for saving user and file metadata.
        fs (AbstractFileSystem): File system instance for saving files.
        authorized (bool, optional): Whether the files are authorized. Defaults to True.

    Returns:
        Tuple[User, List[FileMetadata]]: The created user object and a list of file metadata objects created.
    """
    unvalidated_user = UnvalidatedUser(**user_data)
    print(unvalidated_user)
    user: User = db.save_user(unvalidated_user)
    metadata: List[FileMetadata] = []

    for file_info in files:
        file_path = fs.build_user_path(user, FilePathType.USER_ID_BASE)
        path = fs.create_folder(file_path)
        file_content = file_info["content"]
        file = fs.create_file(
            file_path, file_content, file_info["name"], authorized=authorized
        )
        # Save file metadata to database
        unvalidated_metadata = UnvalidatedFileMetadata(
            user_id=user.id, file_name=file_info["name"], file_path=file.uri
        )
        metadata.append(db.save_file_metadata(unvalidated_metadata))

    return user, metadata

In [6]:
# | export

async def transcribe_file_and_extract_speakers(
    file_metadata: FileMetadata,
    db: AbstractDatabase,
    audio_editor: AudioEditor,
) -> Tuple[Transcription, List[str]]:
    """
    Transcribe an audio file, extract speaker names, and save the transcription to the database.

    Args:
        file_metadata (FileMetadata): The metadata of the file to transcribe.
        db (AbstractDatabase): Database instance for saving transcription data.
        audio_editor (AssemblyAIClient): AssemblyAI client instance for transcription.

    Returns:
        Tuple[Transcription, List[str]]: The created transcription object and a list of extracted speaker names.
    """
    transcription_result: AssemblyAiTranscript = await audio_editor.transcribe(
        str(file_metadata.file_path)
    )
    utterances: List[UnvalidatedUtterance] = transcription_result.utterances
    speaker_names: List[str] = list(
        set([utterance.speaker for utterance in utterances])
    )

    # Future - update speaker names in the transcript using AI + find and replace

    # Save transcription to the database
    unvalidated_transcription = UnvalidatedTranscription(
        file_id=file_metadata.id, text=transcription_result.text
    )
    transcription: Transcription = db.save_transcription(
        unvalidated_transcription, utterances
    )

    return transcription, speaker_names

In [7]:
# | export
async def create_and_save_schema(
    text: str, db: AbstractDatabase, ai_model_client: AIModelClient
) -> Schema:
    """
    Create a schema from the given text using an AI model and save it to the database.

    Args:
        text (str): The text to create a schema from.
        db (AbstractDatabase): The database instance.
        ai_model_client (AIModelClient): The AI model client instance.

    Returns:
        Schema: The saved schema object.
    """
    questions = await ai_model_client.create_schema(text)
    unvalidated_schema = UnvalidatedSchema(
        input_text=text, json_schema=questions.model_dump()
    )
    schema = db.save_schema(unvalidated_schema)
    return schema

In [8]:
# | export

import asyncio

async def extract_info_from_transcriptions(
    schema: Schema,
    transcriptions: Sequence[Transcription],
    db: AbstractDatabase,
    ai_model_client: AIModelClient,
) -> Sequence[Answers]:
    """
    Extract information from transcriptions based on a schema and yields each extraction.

    Args:
        schema (Schema): The schema to use for extraction. Converted to questions internally.
        transcriptions (Sequence[Transcription]): The list of transcriptions to extract information from.
        db (AbstractDatabase): The database instance.
        ai_model_client (AIModelClient): The AI model client instance.

    Yields:
        Iterator[Tuple[Transcription, dict]]: An iterator of tuples containing the transcription and extracted answers.
    """
    questions = Questions(**schema.json_schema)


    tasks = [ai_model_client.extract_information(transcription.text, questions) for transcription in transcriptions]

    # Use asyncio.gather to run them concurrently and wait for all to complete
    answers: list[Answers] = await asyncio.gather(*tasks)

    return answers

In [11]:
# | export

async def get_relevant_utterances_from_query(query: str, transcripts: List[Transcription], db: AbstractDatabase) -> List[Utterance]:
    """
    Returns time-sorted utterances from a query
    """
    if transcripts is None:
        raise ValueError("transcripts cannot be None")
    #allow for several clips from each transcript
    clips_requested = int(max(len(transcripts)*2, 50))
    search_engine = SearchEngine(seconds_buffer=8, similarity_top_k=clips_requested, db=db) 
    utterances = await search_engine.get_utterances_from_query(query, transcripts)

    return utterances



In [10]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore